In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
train_df = pd.read_csv("data/train.csv",encoding="utf-8")
test_df = pd.read_csv("data/test.csv",encoding="utf-8")

In [4]:
train_df.shape

(891, 12)

In [5]:
train_df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [6]:
test_df.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [7]:
test_df.shape

(418, 11)

In [8]:
train_df["train_col"] = 1
test_df["train_col"] = 0

In [9]:
merged_df = pd.concat([train_df,test_df],axis=0)

In [10]:
merged_df.shape

(1309, 13)

In [11]:
merged_df.isna().sum()

PassengerId       0
Survived        418
Pclass            0
Name              0
Sex               0
Age             263
SibSp             0
Parch             0
Ticket            0
Fare              1
Cabin          1014
Embarked          2
train_col         0
dtype: int64

In [34]:
age_df = merged_df.copy()

In [35]:
age_df = age_df.drop(["Survived","Cabin"],axis=1)

In [36]:
age_df.dropna(subset=["Fare","Embarked"],inplace=True)

In [37]:
age_df.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age            263
SibSp            0
Parch            0
Ticket           0
Fare             0
Embarked         0
train_col        0
dtype: int64

In [38]:
get_name = age_df["Name"].apply(lambda x: x.split(",")[1])
get_title = get_name.apply(lambda x: x.split(".")[0])
age_df["Title"] = get_title

In [39]:
age_df["Title"].value_counts()

Title
Mr              756
Miss            259
Mrs             196
Master           61
Rev               8
Dr                8
Col               4
Ms                2
Major             2
Mlle              2
Don               1
Mme               1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Dona              1
Name: count, dtype: int64

In [40]:
title_map = {
    "Mr": "Mr",
    "Miss": "Miss",
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mrs": "Mrs",
    "Mme": "Mrs",
    "Master": "Master",
    "Rev": "Rare",
    "Dr": "Rare",
    "Col": "Rare",
    "Major": "Rare",
    "Don": "Rare",
    "Dona": "Rare",
    "Lady": "Rare",
    "Sir": "Rare",
    "Capt": "Rare",
    "the Countess": "Rare",
    "Jonkheer": "Rare",
}

age_df["Title"] = age_df["Title"].str.strip().map(title_map)

In [41]:
age_df.head(5)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,train_col,Title
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,1,Mr
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,1,Mrs
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,1,Miss
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,1,Mrs
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,1,Mr


In [42]:
age_df.columns

Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Embarked', 'train_col', 'Title'],
      dtype='str')

In [43]:
from sklearn.preprocessing import OrdinalEncoder

or_cols = ['Title','Embarked','Sex','Ticket']

scalers = {}
for col in or_cols:
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    age_df[col] = enc.fit_transform(age_df[[col]])

age_df

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,train_col,Title
0,1,3,"Braund, Mr. Owen Harris",1.0,22.0,1,0,718.0,7.2500,2.0,1,2.0
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0.0,38.0,1,0,814.0,71.2833,0.0,1,3.0
2,3,3,"Heikkinen, Miss. Laina",0.0,26.0,0,0,912.0,7.9250,2.0,1,1.0
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0.0,35.0,1,0,64.0,53.1000,2.0,1,3.0
4,5,3,"Allen, Mr. William Henry",1.0,35.0,0,0,647.0,8.0500,2.0,1,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",1.0,NaN,0,0,709.0,8.0500,2.0,0,2.0
414,1306,1,"Oliva y Ocana, Dona. Fermina",0.0,39.0,0,0,832.0,108.9000,0.0,0,4.0
415,1307,3,"Saether, Mr. Simon Sivertsen",1.0,38.5,0,0,870.0,7.2500,2.0,0,2.0
416,1308,3,"Ware, Mr. Frederick",1.0,NaN,0,0,578.0,8.0500,2.0,0,2.0


In [44]:
age_con = age_df["Age"].isna()

age_na = age_df[age_con]
age_not_na = age_df[~age_con]

In [48]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

iners = []
sil_scores = []

features = ["Pclass","Sex","Age","SibSp","Parch","Ticket","Embarked","Title"]
data = age_not_na[features]

r = range(1,40)
for i in r:
    model = KMeans(n_clusters=i)
    model.fit(data)
    iner = model.inertia_
    sil_score = silhouette_score(data, model.predict(data))
    sil_scores.append(sil_score)
    iners.append(iner)

ValueError: Number of labels is 1. Valid values are 2 to n_samples - 1 (inclusive)

In [17]:
age_not_na

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,train_col
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,1
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,1
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,1
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,1
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,1
...,...,...,...,...,...,...,...,...,...,...,...
409,1301,3,"Peacock, Miss. Treasteall",female,3.0,1,1,SOTON/O.Q. 3101315,13.7750,S,0
411,1303,1,"Minahan, Mrs. William Edward (Lillian E Thorpe)",female,37.0,1,0,19928,90.0000,Q,0
412,1304,3,"Henriksson, Miss. Jenny Lovisa",female,28.0,0,0,347086,7.7750,S,0
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C,0
